# Data understanding

Two corpora from the music21 corpus; each file keeps its encoder's terms ([corpus note](https://github.com/cuthbertLab/music21/blob/master/music21/corpus/license.txt)).

| Corpus | Edition | Licence |
|---|---|---|
| [Bach chorales](https://github.com/cuthbertLab/music21/tree/master/music21/corpus/bach) | 371, numbered as in [Riemenschneider](https://imslp.org/wiki/371_Harmonized_Chorales_and_69_Chorale_Melodies_%28Bach,_Johann_Sebastian%29) | none stated, except 3 kern files (© 1994 CCARH) |
| [Palestrina masses](https://github.com/cuthbertLab/music21/tree/master/music21/corpus/palestrina) | *Le Opere Complete* (Rome: Fratelli Scalera), encoded by John Miller | © 2000 John Miller, derivative formats reserved |


In [ ]:
import json, os, urllib.request
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from music21 import converter, meter, key
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from concurrent.futures import ProcessPoolExecutor

In [ ]:
TAG = "v10.5.0"   # the music21 version in environment.yml
API = f"https://api.github.com/repos/cuthbertLab/music21/git/trees/{TAG}:music21/corpus"
FILES = f"https://raw.githubusercontent.com/cuthbertLab/music21/{TAG}/music21/corpus"
RAW = Path("../data/raw")
INTERIM = Path("../data/interim")   # written by one notebook, read by the next
WORKERS = os.cpu_count() // 2 # Use half of the available CPU cores for parallel processing

In [ ]:
from figures import colours, ink, muted, grid, COLUMN, PAGE, save, boxes, stacked   # the house style, shared with 03_description
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}

In [ ]:
for name in ["bach", "palestrina"]:
    listing = json.load(urllib.request.urlopen(f"{API}/{name}"))
    assert not listing["truncated"]
    out = RAW / name
    out.mkdir(parents=True, exist_ok=True)
    files = [e["path"] for e in listing["tree"] if e["type"] == "blob"]
    missing = [f for f in files if not (out / f).exists()]
    for f in tqdm(missing, desc=name):
        urllib.request.urlretrieve(f"{FILES}/{name}/{f}", out / f)
    print(f"{name}: {len(files)} files, {len(missing)} downloaded")


In [ ]:
def row_of(path):
    score = converter.parse(path)
    time_sigs = score.recurse().getElementsByClass(meter.TimeSignature)
    firsts = [ks[0].sharps for part in score.parts if (ks := part.recurse().getElementsByClass(key.KeySignature))]
    return {
        "corpus": path.parent.name,
        "file": path.name,
        "format": path.suffix[1:],
        "parts": len(score.parts),
        "measures": len(score.parts[0].getElementsByClass("Measure")),
        "duration": float(score.highestTime),   # in quarter notes
        "notes": len(score.recurse().notes),    # notes and chords, each counted once
        "time_signature": ", ".join(dict.fromkeys(t.ratioString for t in time_sigs)) or None,   # every meter in the file, in order
        "key_signature": max(set(firsts), key=firsts.count) if firsts else None,   # the one most parts share: a wrong one in a single part is outvoted; sharps > 0, flats < 0
    }

paths = sorted(p for name in ["bach", "palestrina"] for p in (RAW / name).iterdir())
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    rows = list(tqdm(pool.map(row_of, paths, chunksize=8), total=len(paths), desc="parsing"))

In [ ]:
pieces = pd.DataFrame(rows)
INTERIM.mkdir(parents=True, exist_ok=True)
pieces.to_csv(INTERIM / "00_pieces.csv", index=False)   # one row per file, read by 01_filtering

## Basic metrics comparison

One row per file, read as written, nothing transformed:

| Column | Meaning |
|---|---|
| `parts` | staves in the score (voices or instruments) |
| `measures` | bars in the first part, a pickup bar included |
| `duration` | length in quarter notes, not seconds |
| `notes` | note events; a chord counts once |
| `time_signature` | every meter in the file, in order of appearance |
| `key_signature` | the one most parts start with, as sharps (+) or flats (−) |


### Length

- **Measures:** bars in the first part, pickup included.
- **Duration:** quarter notes, from the start of the score to the end of the last note.
- **Notes:** note events in all parts; a chord counts once.
- **Plot:** log axis; curves give the % of files per tenth of a decade (×1.26); boxes the median and quartiles, whiskers 1.5 times the box, circles the files beyond.

In [ ]:
medians = pieces.groupby("corpus")[["measures", "duration", "notes"]].median()
boxes(pieces, {"measures": "measures", "duration": "duration (quarter notes)", "notes": "notes"},
      "Length of the files on a log scale: curves give the share of files per tenth of a decade, boxes the median and quartiles, "
      "circles the files beyond 1.5 times the box. Palestrina is longer by every measure: "
      f"median {medians.loc['palestrina', 'measures']:g} bars against {medians.loc['bach', 'measures']:g}.", stem="00_size")

**Result**
- Palestrina is longer by every measure: median 39 bars against 15, 304 quarter notes against 48, 448 notes against 223.
- Bars compare best (four beats in both); in quarter notes the ratio rises to about six, because Palestrina's beat is a half note.
- Bach's outliers on the right: files with 8 to 15 parts (8 of the 10 longest).

### Length against rhythmic activity

- **Notes per voice per beat:** notes / parts / beats; the beat is a quarter note in Bach, a half note in Palestrina.
- **Plot:** one dot per file; large dots are the medians.

In [ ]:
BEAT = {"bach": 1, "palestrina": 2}   # the beat in quarter notes: a quarter in 4/4 and 3/4, a half note in 4/2 and 3/2
pieces["activity"] = pieces["notes"] / pieces["parts"] / (pieces["duration"] / pieces["corpus"].map(BEAT))

fig, ax = plt.subplots(figsize=(COLUMN, 2.6))
for name, g in pieces.groupby("corpus"):
    ax.scatter(g["measures"], g["activity"], s=4, color=colours[name], alpha=0.35, linewidths=0, label=name.capitalize())
    ax.scatter(g["measures"].median(), g["activity"].median(), s=40, color=colours[name], edgecolor="white", linewidth=1, zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
lo, hi = pieces["activity"].min(), pieces["activity"].max()
ax.set_yticks([m * 10 ** e for e in range(-2, 2) for m in (1, 2, 5) if lo <= m * 10 ** e <= hi])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:g}")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:g}")
ax.yaxis.set_minor_formatter(lambda v, _: "")
ax.set_xlabel("measures", color=muted)
ax.set_ylabel("notes per voice per beat", color=muted)
ax.grid(color=grid, linewidth=0.4)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(muted)
ax.tick_params(which="both", colors=muted, length=0)
legend = ax.legend(frameon=False, labelcolor=ink, markerscale=3, loc="upper right")
for handle in legend.legend_handles:
    handle.set_alpha(1)
fig.tight_layout(pad=0.3)
save(fig, "00_activity")
plt.show()
activity = pieces.groupby("corpus")["activity"].median()
print("Length against rhythmic activity, one dot per file and the medians as large dots, both axes on a log scale. "
      f"Bach is short with busy voices (median {activity['bach']:.2f} notes per voice per beat), "
      f"Palestrina long with slow ones ({activity['palestrina']:.2f}).")
display(pieces.groupby("corpus")["activity"].describe().round(2).T)

**Result**
- Two clouds: Bach short with busy voices, Palestrina long with slow ones.
- Medians 1.09 and 0.63 notes per voice per beat; the middle halves (1.00–1.21, 0.58–0.69) do not overlap.

### Parts

In [ ]:
voices = pd.cut(pieces["parts"], [0, 3, 4, 5, 6, 99], labels=["2–3", "4", "5", "6", "7+"])
order = ["7+", "6", "5", "4", "2–3"]   # parts, from most to fewest
shares = 100 * pd.crosstab(pieces["corpus"], voices, normalize="index")[order]
stacked(shares, "share of files", f"Parts per score, from most (dark) to fewest (light). {shares.loc['bach', '4']:.1f}% of the Bach files "
        "have four parts; Palestrina spreads over four, five and six voices.", stem="00_parts")
display(pd.crosstab(pieces["parts"], pieces["corpus"]))

**Result**
- Bach: 89% of the files have four parts; the other 45 (2 to 15 parts) need a filter.
- Palestrina: 3 to 8 voices, mostly 4 (38%) or 5 (36%).

### Texture

- **Measure:** in the four-voice files, how many voices sing at each sixteenth note.
- **Left out:** moments when every voice rests.

In [ ]:
def sounding_of(path):
    score = converter.parse(path)
    grid = np.arange(0, float(score.highestTime), 0.25)   # sixteenth notes
    sounding = np.zeros((len(score.parts), len(grid)), dtype=bool)
    for v, part in enumerate(score.parts):
        for n in part.flatten().notes:
            sounding[v, (grid >= float(n.offset)) & (grid < float(n.offset + n.quarterLength))] = True
    return np.bincount(sounding.sum(axis=0), minlength=5)   # time steps with 0, 1, 2, 3, 4 voices singing

four = pieces[pieces["parts"] == 4]
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    counts = list(pool.map(sounding_of, [RAW / c / f for c, f in zip(four["corpus"], four["file"])], chunksize=8))
totals = pd.DataFrame(counts, index=four["corpus"]).groupby(level=0).sum()   # time steps per corpus, by voices singing
rest = (100 * totals[0] / totals.sum(axis=1)).round(1)   # every voice silent: a pause of the whole texture, left out below
shares = pd.DataFrame({"4": totals[4], "3": totals[3], "1–2": totals[1] + totals[2]})
shares = 100 * shares.div(shares.sum(axis=1), axis=0)
stacked(shares, "share of time", "How many of the four voices sing at once, in the four-voice files; moments when every voice rests are "
        f"left out. All four sing {shares.loc['bach', '4']:.1f}% of the time in Bach and {shares.loc['palestrina', '4']:.1f}% in Palestrina.", stem="00_texture")
display(shares.round(1).assign(files=four.groupby("corpus").size(), rest=rest).T)

**Result**
- All four voices sing 99.8% of the time in Bach, 49.2% in Palestrina; two or fewer, 13.2% in Palestrina.
- So a model of Palestrina must generate rests and a changing number of sounding notes.
- Every voice rests 1.0% of Bach's time, 0.1% of Palestrina's.

### Time signature

In [ ]:
kind = {"4/4": "duple", "2/2": "duple", "4/2": "duple", "3/4": "triple", "3/2": "triple"}   # anything else: compound, or a change of meter
pieces["meter"] = pieces["time_signature"].map(kind).fillna("other")
shares = 100 * pd.crosstab(pieces["corpus"], pieces["meter"], normalize="index")[["duple", "triple", "other"]]
labels = pieces.groupby(["corpus", "meter"])["time_signature"].agg(lambda s: s.mode()[0]).unstack()   # the signature that stands for each block
stacked(shares, "share of files", "Time signature over the whole file: duple, triple, or other (compound or changing). Both corpora "
        f"are mostly duple: Bach in 4/4 ({shares.loc['bach', 'duple']:.1f}%), Palestrina in 4/2 or 2/2 ({shares.loc['palestrina', 'duple']:.1f}%).", labels, stem="00_meter")
display(pd.crosstab(pieces["time_signature"], pieces["corpus"]))

**Result**
- About nine files in ten are in duple time: Bach in 4/4 (87.7%), Palestrina in 4/2 (91.4%; one file in 2/2).
- The rest is mostly triple (3/4, 3/2); six Bach files are neither: five change meter, one is in 12/8.
- Same bar, different beat: a quarter note in Bach, a half note in Palestrina.

### Key signature

In [ ]:
side = np.sign(pieces["key_signature"]).map({-1: "flats", 0: "none", 1: "sharps"})
shares = 100 * pd.crosstab(pieces["corpus"], side, normalize="index")[["sharps", "none", "flats"]]
labels = (pieces.groupby(["corpus", side])["key_signature"]
          .agg(lambda s: f"{s.min()}" if s.min() == s.max() else f"{s.min()} to {s.max()}").unstack())   # the signatures in each block
low, high = pieces.groupby("corpus")["key_signature"].min(), pieces.groupby("corpus")["key_signature"].max()
stacked(shares, "share of files", f"Key signature, sharps positive and flats negative. Bach spreads from {-low['bach']} flats to "
        f"{high['bach']} sharps; Palestrina has no flat or one flat, the two systems of modal polyphony.", labels, stem="00_key_signature")
display(pd.crosstab(pieces["key_signature"], pieces["corpus"]))

**Result**
- Bach: from four flats to four sharps.
- Palestrina: no flat (63%) or one flat (37%), *cantus durus* and *cantus mollis*, the two systems of modal polyphony.

### Rhythmic grid

- **Why:** the model reads a grid of equal steps; a note between two grid lines would have to move.
- **Measure:** per step, from the thirty-second to the whole note: notes on a grid line (kept, solid), files with a note off it (touched, dashed).
- **Marked:** the coarsest grid that keeps 99.9% of the notes.

In [ ]:
def notes_of(path):   # (onset, duration) of every note, in quarter notes from the start of the score
    score = converter.parse(path)
    return [(float(n.offset), float(n.quarterLength)) for part in score.parts for n in part.flatten().notes]

with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    notes = dict(zip(paths, pool.map(notes_of, paths, chunksize=8)))

GRIDS = {"thirty-second": 0.125, "sixteenth": 0.25, "eighth": 0.5, "quarter": 1, "half": 2, "whole": 4}   # step in quarter notes
SYMBOLS = ["\U0001D162", "\U0001D161", "\U0001D160", "\U0001D15F", "\U0001D15E", "\U0001D15D"]   # the same note values, as glyphs
KEEP = 99.9   # the share of notes a grid must keep
DASHED = dict(linestyle=(0, (3, 1.6)), dash_capstyle="round")   # short dashes for the files touched

def on_grid(onsets, step):
    return np.isclose(onsets / step, np.round(onsets / step))

kept = {}
for corpus in colours:
    files = [np.array([o for o, _ in ns]) for p, ns in notes.items() if p.parent.name == corpus]
    every = np.concatenate(files)
    kept[corpus, "notes"] = {g: 100 * on_grid(every, step).mean() for g, step in GRIDS.items()}
    kept[corpus, "files"] = {g: 100 * np.mean([on_grid(f, step).all() for f in files]) for g, step in GRIDS.items()}   # no note moved
kept = pd.DataFrame(kept).T   # rows: (corpus, notes or files); columns: grid steps

x = np.arange(len(GRIDS))
marked = {}
fig, axes = plt.subplots(2, 1, figsize=(COLUMN, 3.4), sharex=True)
for ax, name in zip(axes, colours):
    c, notes_kept, touched = colours[name], kept.loc[(name, "notes")].to_numpy(), 100 - kept.loc[(name, "files")].to_numpy()
    chosen = max(i for i, v in enumerate(notes_kept) if v >= KEEP)   # the coarsest grid that keeps enough notes
    marked[name] = list(GRIDS)[chosen]
    ax.plot(x, notes_kept, color=c, linewidth=1.2)
    ax.plot(x, touched, color=c, linewidth=1.2, **DASHED)
    ax.scatter([chosen] * 2, [notes_kept[chosen], touched[chosen]], s=28, color=c, edgecolor="white", linewidth=1, zorder=3)
    ax.set_ylim(-5, 105)
    ax.set_yticks([0, 50, 100], ["0%", "50%", "100%"])
    ax.set_xlim(-0.5, len(GRIDS) - 0.5)
    ax.grid(color=to_rgba(grid, 0.6), linewidth=0.4)   # faint grid both ways, as in the other figures
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(muted)
    ax.tick_params(colors=muted, length=0)
axes[0].spines["bottom"].set_visible(False)   # the two panels share the x axis drawn under the bottom one
axes[1].set_xticks(x, SYMBOLS, fontfamily="Noto Music", fontsize=12, color=ink)   # Noto Music has the note glyphs, Latin Modern does not
axes[1].set_xlabel("grid step", color=muted)
fig.legend(handles=[Line2D([], [], linestyle="", marker="o", markersize=6, markerfacecolor=colours[name], markeredgecolor="white",
                           label=name.capitalize()) for name in colours],   # dots, like the marks in the plot
           loc="upper left", ncols=2, frameon=False, labelcolor=ink, handlelength=0.8, handletextpad=0.4)
fig.legend(handles=[Line2D([], [], color=muted, linewidth=1.2, label="notes kept"),
                    Line2D([], [], color=muted, linewidth=1.2, label="files touched", **DASHED)],
           loc="upper right", ncols=2, frameon=False, labelcolor=ink, handlelength=2, handletextpad=0.4, columnspacing=1)
fig.tight_layout(rect=(0, 0, 1, 0.94), pad=0.3, h_pad=1.5)   # room for the legends
save(fig, "00_grid")
plt.show()
print("Share of notes kept on the grid and of files touched by it, for each grid step. The dots mark the coarsest grid that keeps "
      f"{KEEP:g}% of the notes, a {marked['bach']} note for Bach and an {marked['palestrina']} note for Palestrina, where "
      f"{100 - kept.loc[('bach', 'files'), marked['bach']]:.1f}% and {100 - kept.loc[('palestrina', 'files'), marked['palestrina']]:.1f}% "
      "of the files are touched.")
display(kept.round(2).T)

shortest = {corpus: min(d for p, ns in notes.items() if p.parent.name == corpus for _, d in ns if d > 0) for corpus in colours}
display(pd.Series(shortest, name="shortest note, in quarter notes").to_frame())
off = pd.DataFrame([(p.parent.name, p.name, d) for p, ns in notes.items() for o, d in ns if not np.isclose(o * 8, round(o * 8))],
                   columns=["corpus", "file", "duration"])   # onsets on no grid down to the thirty-second
display(off.groupby(["corpus", "file"])["duration"].agg(["size", "median"]).round(3))

**Result**
- Files are touched long before notes are lost: on an eighth-note grid Bach keeps 97.1% of its notes but touches 69.0% of its files; on a quarter-note grid Palestrina keeps 99.3% and touches 80.3%.
- Marked: a sixteenth for Bach, an eighth for Palestrina, a quarter of the beat in both. They touch 8 Bach files (thirty-seconds) and 3 Palestrina files (264 triplet notes, 4/3 of a quarter).

## Groups of files

- **Why:** movements belong to masses and chorales share melodies; a split for the model keeps a group on one side.

### Palestrina: masses

In [ ]:
def mass_of(path):
    return next(line.split(":", 1)[1].strip() for line in open(path, encoding="latin-1")   # the kern files are Latin-1
                if line.startswith("!!!OPR"))

palestrina = pieces[pieces["corpus"] == "palestrina"].copy()
palestrina["mass"] = [mass_of(RAW / "palestrina" / f) for f in palestrina["file"]]
palestrina["section"] = palestrina["file"].str.split("_|\\.").str[0]   # Kyrie, Gloria, Credo, Sanctus, Benedictus, Agnus
per_mass = palestrina.groupby("mass").size()
display(per_mass.describe().round(1).to_frame("movements per mass"))

**Result**
- 1,318 movements from 101 masses, 6 to 26 files each (median 12); each section is split into one to four files, the Credo into about four.

In [ ]:
usual = palestrina.groupby("mass")["parts"].agg(lambda s: s.mode().max())   # the voice count most movements of the mass use
change = palestrina["parts"] - palestrina["mass"].map(usual)
palestrina["voices"] = pd.cut(change, [-10, -1, 0, 10], labels=["fewer voices", "same", "more voices"])
table = pd.crosstab(palestrina["section"], palestrina["voices"]).loc[["Kyrie", "Gloria", "Credo", "Sanctus", "Benedictus", "Agnus"]]
shares = table.div(table.sum(axis=1), axis=0) * 100

c = colours["palestrina"]
fig, ax = plt.subplots(figsize=(COLUMN, 2.1))
left = np.zeros(len(shares))
for voices, colour in zip(shares.columns, [to_rgba(c, 0.4), grid, c]):
    ax.barh(shares.index, shares[voices], left=left, color=colour, edgecolor="white", label=voices)
    left += shares[voices].to_numpy()
ax.invert_yaxis()
ax.xaxis.set_major_formatter(lambda v, _: f"{v:g}%")
ax.legend(frameon=False, labelcolor=ink, ncols=3, loc="lower left", bbox_to_anchor=(0, 1), columnspacing=1, handlelength=1)
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
ax.tick_params(colors=muted, length=0)
ax.tick_params(axis="y", colors=ink)
fig.tight_layout(pad=0.3)
save(fig, "00_mass_voices")
plt.show()
print("Voices of each Palestrina movement against the usual voices of its mass, by section. Only the Agnus adds voices "
      f"({shares.loc['Agnus', 'more voices']:.0f}% of its files); the Benedictus drops them most often "
      f"({shares.loc['Benedictus', 'fewer voices']:.0f}%).")
display(usual.value_counts().sort_index().rename_axis("usual voices").rename("masses").to_frame())
display(table)

**Result**
- Masses for 4 voices (39), 5 (37), 6 (21), 8 (4).
- The count follows the section: the Benedictus (82 of 160 files) and the Credo (109 of 392) drop voices; only the Agnus adds them (63 of 193).
- So a mass has no fixed number of voices.

### Bach: melodies

In [ ]:
def melody_of(path):
    soprano = converter.parse(path).parts[0].stripTies().flatten().notes
    pitches = [n.pitches[-1].midi for n in soprano]   # top note of a chord
    return tuple(b - a for a, b in zip(pitches, pitches[1:]))   # intervals: the same tune in any key

bach = pieces[pieces["corpus"] == "bach"].copy()
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    bach["melody"] = list(pool.map(melody_of, [RAW / "bach" / f for f in bach["file"]], chunksize=8))
per_melody = bach.groupby("melody").size()
shared = per_melody[per_melody > 1]
display(pd.DataFrame({"files": [len(bach)], "distinct melodies": [len(per_melody)], "repeated melodies": [len(shared)],
                      "files with a shared soprano": [shared.sum()], "share": [f"{100 * shared.sum() / len(bach):.1f}%"]}, index=["Bach"]).T)

**Result**
- Matched as intervals, in any key: 18 soprano lines appear twice; 36 files (8.7% of Bach) share their soprano.
- A lower bound: an exact match misses variants of a tune. Grouping by tune uses a looser match, in 02_corpus.